# Deepfake Detection — Kaggle GPU Runner

Trains the cross-dataset deepfake detectors on a free Kaggle GPU (T4 ×2 or P100, 16 GB).

**Before running:** turn on the GPU (Notebook ▸ Settings ▸ Accelerator ▸ GPU) and attach your uploaded dataset
(Add Input ▸ your `deepfake-project` dataset). See `CLOUD_SETUP.md` for how to upload.


## 1. Check GPU


In [ ]:
!nvidia-smi

## 2. Locate the attached dataset

Edit `DATASET` if your input folder is named differently. The dataset must contain `faces/`, `ml/`, and `experiments/`.


In [ ]:
import os, glob, sys, subprocess

# Diagnostic: show what is actually mounted
print('Under /kaggle/input:')
for c in sorted(glob.glob('/kaggle/input/*')):
    sub = sorted(os.listdir(c))[:10] if os.path.isdir(c) else []
    print(' ', c, '->', sub)

# Recursively find the folder that contains BOTH ml/ and faces/ (handles nested zips)
def find_dataset(root='/kaggle/input', max_depth=4):
    hits = []
    for cur, dirs, _ in os.walk(root):
        if cur.count('/') - root.count('/') > max_depth:
            dirs[:] = []; continue
        if os.path.isdir(os.path.join(cur, 'ml')) and os.path.isdir(os.path.join(cur, 'faces')):
            hits.append(cur); dirs[:] = []   # found it; do not descend further
    return sorted(hits)

hits = find_dataset()
assert hits, 'No folder with BOTH ml/ and faces/ under /kaggle/input - see the listing above.'
assert len(hits) == 1, f'Multiple candidates {hits}; detach extra datasets.'
DATASET = hits[0]
print('Dataset:', DATASET)
print('faces/ :', os.path.isdir(os.path.join(DATASET, 'faces')))
print('ml/    :', os.path.isdir(os.path.join(DATASET, 'ml')))


## 2b. Guard — correct, identity-disjoint dataset (C1 regression gate)

Runs **after** auto-detect and **before** any training. Aborts if more than one input dataset is attached (so you cannot silently train on a stale/leaky upload), and asserts the mounted Exp1 faces are identity-disjoint (train∩test == 0).

In [ ]:
# (a) exactly ONE folder with ml/+faces/ anywhere under /kaggle/input
def _find_ds(root='/kaggle/input', max_depth=4):
    hits = []
    for cur, dirs, _ in os.walk(root):
        if cur.count('/') - root.count('/') > max_depth:
            dirs[:] = []; continue
        if os.path.isdir(os.path.join(cur, 'ml')) and os.path.isdir(os.path.join(cur, 'faces')):
            hits.append(cur); dirs[:] = []
    return sorted(hits)
_cands = _find_ds()
print('Inputs with ml/+faces/:', _cands)
assert len(_cands) == 1, (
    f'Expected exactly ONE dataset with ml/+faces/, found {len(_cands)}: {_cands}. '
    'Detach stale/duplicate datasets so training cannot read the wrong faces.')
assert os.path.realpath(_cands[0]) == os.path.realpath(DATASET), (
    f'Auto-detected DATASET ({DATASET}) != the single candidate ({_cands[0]}).')
print('Resolved DATASET:', DATASET)

# (b) Exp1 train-test identity overlap must be 0 (the C1 fix)
_EXP = 'exp1_ffpp_to_ffpp'
_METHODS = ('Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter')

def _ids_from(stem):
    m = re.match(r'^(\d{3})$', stem)                 # real id NNN
    if m:
        return {m.group(1)}
    for meth in _METHODS:                            # classic fake Method_T_S
        if stem.startswith(meth + '_'):
            return {p for p in stem[len(meth) + 1:].split('_')[:2] if p.isdigit()}
    return set()                                     # DeepFakeDetection_*/other -> no FF++ id

def _split_ids(split):
    ids = set()
    for lab in ('real', 'fake'):
        d = os.path.join(DATASET, 'faces', _EXP, split, lab)
        if os.path.isdir(d):
            for f in os.listdir(d):
                mm = re.match(r'(.+?)_frame\d+_face\d+\.jpg$', f)
                if mm:
                    ids |= _ids_from(mm.group(1))
    return ids

_tr, _va, _te = _split_ids('train'), _split_ids('val'), _split_ids('test')
_tt, _vt = len(_tr & _te), len(_va & _te)
print(f'Exp1 identities: train={len(_tr)} val={len(_va)} test={len(_te)} '
      f'| train&test={_tt} val&test={_vt}')
assert _tt == 0 and _vt == 0, (
    f'LEAKAGE: Exp1 train&test={_tt}, val&test={_vt} (must be 0). '
    'You are about to train on a STALE/LEAKY faces upload — re-upload the corrected dataset.')
print('OK: single dataset, Exp1 identity-disjoint. Safe to train.')


## 3. Copy code to a writable folder

`/kaggle/input` is read-only, so copy `ml/` into `/kaggle/working`. Faces stay in the read-only input (read directly).


In [ ]:
import shutil
WORK = '/kaggle/working'
ML   = os.path.join(WORK, 'ml')
if os.path.exists(ML): shutil.rmtree(ML)
shutil.copytree(os.path.join(DATASET, 'ml'), ML)
# experiments/ manifest (used by face extraction only; copy if present)
exp_src = os.path.join(DATASET, 'experiments')
if os.path.isdir(exp_src) and not os.path.exists(os.path.join(WORK,'experiments')):
    shutil.copytree(exp_src, os.path.join(WORK,'experiments'))
FACES   = os.path.join(DATASET, 'faces')
RESULTS = os.path.join(WORK, 'results')
os.makedirs(RESULTS, exist_ok=True)
print('ML     =', ML)
print('FACES  =', FACES)
print('RESULTS=', RESULTS)

## 3b. Apply working-copy fixes

Patches the copied `ml/` for PyTorch 2.6 compatibility (`torch.load` `weights_only=False`) and the progressive-unfreezing learning-rate fix (`0.1`→`0.5`, so the LR no longer collapses to 1e-6). **Idempotent** — it no-ops if the uploaded `ml/` already contains the fixes. Runs every session because cell 3 re-copies `ml/` fresh.

In [ ]:
import pathlib

def _patch(fname, old, new, tag):
    p = pathlib.Path(ML) / fname
    s = p.read_text()
    if old in s:
        p.write_text(s.replace(old, new)); print(f'  patched: {tag}')
    elif new in s:
        print(f'  already fixed: {tag}')
    else:
        print(f'  WARNING: target not found for {tag} ({fname}) -- check manually')

print('Applying working-copy fixes to', ML)
_patch('evaluate.py',
       'torch.load(checkpoint_path, map_location=device)',
       'torch.load(checkpoint_path, map_location=device, weights_only=False)',
       'evaluate.py torch.load weights_only=False (PyTorch 2.6)')
_patch('train.py',
       'lr=LR * (0.1 ** (n_unfreeze // 2))',
       'lr=LR * (0.5 ** (n_unfreeze // 2))',
       'train.py progressive-unfreeze LR (0.1 -> 0.5)')
_patch('train.py',
       'if module is not None:\n            for param in module.parameters():',
       'if isinstance(module, nn.Module):\n            for param in module.parameters():',
       'train.py ViT _freeze_backbone guard (str global_pool)')


## 4. Install dependencies

Kaggle ships torch + most libs; we just ensure `timm` and `grad-cam` are present.


In [ ]:
!pip install -q 'timm>=0.9.12' 'grad-cam>=1.5.0' 'scikit-learn>=1.4.0' 'scipy>=1.12.0' seaborn
import timm, torch
print('timm', timm.__version__, '| cuda', torch.cuda.is_available())

## 5. Sanity run — one experiment

fp16 (AMP) works on Kaggle GPUs, so we keep `USE_AMP=1` (fast). Confirm `val_auc` climbs above 0.5 and a `metrics.json` appears.


In [ ]:
env = dict(os.environ, USE_AMP='1')   # fp16 OK on T4/P100
cmd = [sys.executable, 'run_all.py', '--exp', 'exp1',
       '--faces', FACES, '--results', RESULTS, '--model_preset', 'efficientnet_b4']
subprocess.run(cmd, cwd=ML, env=env, check=True)

In [ ]:
import json
m = json.load(open(os.path.join(RESULTS, 'exp1_ffpp_to_ffpp', 'metrics.json')))
print('video AUC :', m['video_level']['auc'])
print('n_videos  :', m['n_videos'], '| n_faces:', m['n_faces'])

## 6. Full pipeline

Run one model at a time. **Re-run this cell changing `PRESET`** for each baseline:
`efficientnet_b4` → `xception` → `resnet50` → `convnext_tiny` → `vit_base`.

After each model finishes, the next cell snapshots the combined table so it isn't overwritten.


In [ ]:
PRESET = 'efficientnet_b4'   # <-- change this per model and re-run
env = dict(os.environ, USE_AMP='1')
cmd = [sys.executable, 'run_all.py', '--exp', 'all',
       '--faces', FACES, '--results', RESULTS, '--model_preset', PRESET]
subprocess.run(cmd, cwd=ML, env=env, check=True)

# Snapshot the combined table/json (root files get overwritten each run)
for fn in ('results_table.csv','all_results.json'):
    src = os.path.join(RESULTS, fn)
    if os.path.exists(src):
        ext = fn.split('.')[-1]
        shutil.copy(src, os.path.join(RESULTS, f'{fn.split(".")[0]}_{PRESET}.{ext}'))
print('Done:', PRESET)

## 6b. (Recommended) Resumable all-in-one runner

Runs every baseline + ablation in the right order, **per experiment**, and **skips anything already finished**
(detected via an existing `metrics.json`). Kaggle sessions stop after ~9–12 h, so just re-run this one cell in a
fresh session and it continues where it left off. Use this *instead of* manually editing `PRESET` in steps 6–7.

Reorder/trim `JOBS` if you want to prioritise (e.g. EfficientNet-B4 + Xception first).


In [ ]:
import time, sys, os, subprocess
sys.path.insert(0, ML)  # reuse the real path/naming logic, no drift
from run_all import run_dir_name, resolve_model_name, eval_ablation_name, MODEL_PRESETS

EXPS = ['exp1','exp2','exp3','exp4']
EXP_DIRS = {'exp1':'exp1_ffpp_to_ffpp','exp2':'exp2_ffpp_to_celebdf',
            'exp3':'exp3_celebdf_to_ffpp','exp4':'exp4_mixed_to_dfd'}

# (preset, ablation, aggregation, eval_only) — order matters: b4-full first (its
# checkpoints are reused by the eval-only ablations).
JOBS = [
    ('efficientnet_b4', 'full',                    'mean',   False),
    ('xception',        'full',                    'mean',   False),
    ('resnet50',        'full',                    'mean',   False),
    ('convnext_tiny',   'full',                    'mean',   False),
    ('vit_base',        'full',                    'mean',   False),
    ('efficientnet_b4', 'no_label_smoothing',      'mean',   False),
    ('efficientnet_b4', 'no_progressive_unfreeze', 'mean',   False),
    ('efficientnet_b4', 'no_train_augmentation',   'mean',   False),
    ('efficientnet_b4', 'no_tta',                  'mean',   True),
    ('efficientnet_b4', 'full',                    'median', True),
    ('efficientnet_b4', 'full',                    'max',    True),
    ('efficientnet_b4', 'full',                    'topk',   True),
]

env = dict(os.environ, USE_AMP='1')  # fp16 OK on Kaggle GPUs

def expected_dir(preset, ablation, aggregation, exp):
    model_name = resolve_model_name(preset, preset)
    eval_abl = eval_ablation_name(ablation, aggregation)
    return os.path.join(RESULTS, run_dir_name(EXP_DIRS[exp], model_name, eval_abl))

def train_ckpt_dir(preset, ablation, exp):
    # where the checkpoint lives (eval-only jobs need the trained 'full' ckpt)
    model_name = resolve_model_name(preset, preset)
    train_abl = 'full' if ablation == 'no_tta' else ablation
    return os.path.join(RESULTS, run_dir_name(EXP_DIRS[exp], model_name, train_abl))

t_start = time.time()
for preset, ablation, aggregation, eval_only in JOBS:
    tag = f'{preset} | {ablation} | agg={aggregation}' + (' | eval_only' if eval_only else '')
    for exp in EXPS:
        outdir = expected_dir(preset, ablation, aggregation, exp)
        if os.path.exists(os.path.join(outdir, 'metrics.json')):
            print(f'SKIP  {tag}  {exp}  (done)')
            continue
        if eval_only and not os.path.exists(os.path.join(train_ckpt_dir(preset, ablation, exp), 'best_model.pth')):
            print(f'WAIT  {tag}  {exp}  (needs trained checkpoint first) -- skipping for now')
            continue
        args = [sys.executable,'run_all.py','--exp',exp,'--faces',FACES,'--results',RESULTS,
                '--model_preset',preset,'--ablation',ablation,'--aggregation',aggregation]
        if eval_only: args.append('--eval_only')
        print(f'RUN   {tag}  {exp}  [elapsed {(time.time()-t_start)/3600:.1f}h]')
        rc = subprocess.run(args, cwd=ML, env=env).returncode
        if rc != 0:
            print(f'FAIL  {tag}  {exp}  (exit {rc}) -- continuing')
        # snapshot combined tables so they are not overwritten
        for fn in ('results_table.csv','all_results.json'):
            src=os.path.join(RESULTS,fn)
            if os.path.exists(src):
                base,ext=fn.rsplit('.',1)
                import shutil; shutil.copy(src, os.path.join(RESULTS, f'{base}_{preset}_{ablation}_{aggregation}.{ext}'))
print(f'\nAll jobs processed. Total elapsed {(time.time()-t_start)/3600:.1f}h')

## 7. Ablations (EfficientNet-B4)

Training ablations retrain; eval ablations reuse the trained full checkpoints from step 6 (run those first).


In [ ]:
env = dict(os.environ, USE_AMP='1')
def run(args):
    subprocess.run([sys.executable,'run_all.py','--faces',FACES,'--results',RESULTS]+args, cwd=ML, env=env, check=True)

# Training ablations
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_label_smoothing'])
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_progressive_unfreeze'])
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_train_augmentation'])
# Eval-only ablations (need step 6 efficientnet_b4 checkpoints first)
run(['--exp','all','--model_preset','efficientnet_b4','--ablation','no_tta','--eval_only'])
for agg in ('median','max','topk'):
    run(['--exp','all','--model_preset','efficientnet_b4','--aggregation',agg,'--eval_only'])

## 8. Download results

Everything under `/kaggle/working/results` (checkpoints, metrics.json, failure_cases.csv, plots) is saved with the notebook output and can be downloaded. This zips it for one-click download from the notebook's **Output** tab.


In [ ]:
shutil.make_archive('/kaggle/working/results_bundle', 'zip', RESULTS)
print('Created /kaggle/working/results_bundle.zip')
print('Size MB:', round(os.path.getsize('/kaggle/working/results_bundle.zip')/1e6,1))